# Explore Dataset — candidate `class_characteristic`

For every raw GEO series in `configs/dataset_origin.yaml` that still has
`class_characteristic: ""` (a TODO entry), this notebook parses the series
matrix and ranks every `!Sample_characteristics_ch1` column as a candidate
class label — how many classes it has, how many samples are NA, and the
smallest class size — so you can pick a clean column and fill it in.

Run this **before** `00.a_Format_GEO_Datasets` for any origin still marked
TODO. Requires the origin's `series_matrix` file to already be downloaded.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ directory.
REPO_ROOT = Path("..").resolve()          # repo root (parent of notebooks/)
sys.path.insert(0, str(REPO_ROOT))        # make src.* importable

import pandas as pd
from src.helper import ConfigLoader
from src.dataset_builder import parse_series_matrix, discover_class_characteristics

cfg = ConfigLoader(config_root=str(REPO_ROOT / "configs"))
print("Repo root:", REPO_ROOT)

## 2. Collect origins that need exploring

Every entry in `configs/dataset_origin.yaml` with `class_characteristic: ""`
— skipping any whose `series_matrix` file isn't downloaded yet.

In [ ]:
origins_cfg = cfg.load_dataset_origin_config()["origins"]

todo = []
for gse_id, oc in origins_cfg.items():
    if oc.get("class_characteristic"):
        continue   # already configured — nothing to explore
    origin = cfg.get_origin_config(gse_id)
    sm_path = Path(origin["series_matrix"])
    todo.append({
        "gse_id": gse_id,
        "tissue": origin["tissue"],
        "series_matrix": sm_path,
        "downloaded": sm_path.exists(),
    })

status_df = pd.DataFrame(todo)
if status_df.empty:
    print("No TODO origins — every entry already has a class_characteristic set.")
else:
    missing = status_df[~status_df["downloaded"]]
    if not missing.empty:
        print("Not yet downloaded (skipping):", ", ".join(missing["gse_id"]))
display(status_df)

## 3. Rank candidate class characteristics for each origin

For every downloaded TODO origin: parse the series matrix, rank every
characteristic column (0-NA first, then larger smallest-class, then fewer
classes), and print the full value counts for **every** candidate column —
not just the top-ranked one — so you can compare them side by side.

In [ ]:
MAX_CLASSES_TO_PRINT = 30   # skip dumping full value counts for ID-like columns (e.g. 'title')

explored = {}   # gse_id -> (series_meta, ranking_df)

for spec in todo:
    if not spec["downloaded"]:
        continue

    gse_id, sm_path = spec["gse_id"], spec["series_matrix"]
    print(f"\n{'=' * 70}\n{gse_id} — {spec['tissue']}\n{'=' * 70}")

    _, sample_meta, series_meta = parse_series_matrix(sm_path)
    print(f"{series_meta['geo_accession']} — {series_meta['title']}")
    print(f"Platform: {series_meta['platform_id']} | samples: {len(sample_meta)}\n")

    ranking = discover_class_characteristics(sample_meta)
    explored[gse_id] = (series_meta, ranking)

    print("Candidate class columns (ranked: 0-NA first, then larger smallest-class):")
    display(ranking.drop(columns="classes"))

    for _, cand in ranking.iterrows():
        col = cand["characteristic"]
        print(f"\n-- '{col}' ({cand['n_classes']} classes, {cand['na_count']} NA) --")
        if cand["n_classes"] > MAX_CLASSES_TO_PRINT:
            print(f"   (skipped: {cand['n_classes']} distinct values > "
                  f"{MAX_CLASSES_TO_PRINT} — likely an ID-like column, not a class label)")
            continue
        counts = pd.Series(
            sample_meta[col].astype(str).str.strip()
        ).value_counts()
        print(counts.rename("n_samples").to_string())

## 4. Inspect one characteristic in detail (optional)

Change `GSE_ID` / `CHARACTERISTIC` to preview the raw value counts of any
column for any explored origin before committing it to
`configs/dataset_origin.yaml`.

In [ ]:
GSE_ID = next(iter(explored), None)          # <-- change to inspect another origin
CHARACTERISTIC = None                          # <-- e.g. "tumor_stage"; None = top-ranked

if GSE_ID is None:
    print("Nothing explored yet — run section 3 first.")
else:
    sm_path = next(s["series_matrix"] for s in todo if s["gse_id"] == GSE_ID)
    _, sample_meta, _ = parse_series_matrix(sm_path)
    ranking = explored[GSE_ID][1]
    col = CHARACTERISTIC or (ranking.iloc[0]["characteristic"] if not ranking.empty else None)

    if col is None or col not in sample_meta.columns:
        print(f"Column '{col}' not found. Available: "
              f"{[c for c in sample_meta.columns if c != 'sample_id']}")
    else:
        print(f"{GSE_ID} · '{col}'")
        display(sample_meta[["sample_id", col]])

## 5. Summary — recommended `class_characteristic` per origin

Copy the recommended value into `configs/dataset_origin.yaml`, then run
`00.a_Format_GEO_Datasets` to build the CuMiDa CSVs.

In [ ]:
summary_rows = []
for gse_id, (series_meta, ranking) in explored.items():
    if ranking.empty:
        summary_rows.append({
            "gse_id": gse_id, "platform": series_meta.get("platform_id"),
            "recommended": None, "n_classes": None, "na_count": None,
            "min_class_size": None,
        })
        continue
    top = ranking.iloc[0]
    summary_rows.append({
        "gse_id": gse_id,
        "platform": series_meta.get("platform_id"),
        "recommended": top["characteristic"],
        "n_classes": top["n_classes"],
        "na_count": top["na_count"],
        "min_class_size": top["min_class_size"],
    })

pd.DataFrame(summary_rows)